In [15]:
#import
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../..')


import numpy as np
import src.demo as demo
import src.viz_utils as viz_utils
import src.utils as utils
import importlib
from src.utils import CanonPart, CanonPartMetadata, get_pointcloud_in_cam_frame, transform_cloud_to_base, remove_outliers
from PIL import Image
import torch
import open3d as o3d
import open3d.visualization.gui as gui
import copy as cp
import pickle
import matplotlib

import itertools

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:

# Load the pointcloud files (for visualization purposes)
root_file = '/home/rthomp12/fewshot/scripts/experiment_notebooks/teapot_on_mug_20250114-151255'
save_name = f'{root_file}/init_scene_pcls.npz'
scene_pcls = np.load(save_name)
scene_pcls = {k: scene_pcls[k] for k in scene_pcls.keys()}

# Load recorded demo transformation
transform_name = f'{root_file}/ee_transform.npz'
demo_transform = np.load(transform_name)

init_transform = utils.pos_quat_to_transform(demo_transform['init_pos'],
                                             demo_transform['init_quat'])

final_transform = utils.pos_quat_to_transform(demo_transform['final_pos'],
                                              demo_transform['final_quat'])

ee_transform = np.matmul(final_transform, np.linalg.inv(init_transform))

preplace_transform = utils.pos_quat_to_transform(demo_transform['preplace_pos'],
                                                 demo_transform['preplace_quat'])

inferred_preplace = np.matmul(preplace_transform, np.linalg.inv(final_transform))
ee_preplace = np.matmul(inferred_preplace, ee_transform)
#ee_transform = np.matmul(inferred_preplace, ee_transform)

# Load the warp reconstructions
warps = np.load(f"{root_file}/initial_scene_warps.npz", allow_pickle=True)
warps = {k: warps[k] for k in warps.keys()}

child_params = warps['child_params'].item()
parent_params = warps['parent_params'].item()
child_reconstruction = warps['child_reconstructions'].item()

In [17]:
#Set the object names and canon files that we're using

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~Mug v Rack
# parent_part_names = ['trunk', 'branch']
# child_part_names = ['cup', 'handle']


# parent_part_model_files = {'trunk': '/home/rthomp12/fewshot/syn_rack_easy_trunk_20250110-014541_8', 
#                            'branch': '/home/rthomp12/fewshot/syn_rack_easy_branch_20250110-014541_8'}
# child_part_model_files = {'cup': '/home/rthomp12/fewshot/mug_cup_20250103-220631_5',
#                           'handle': '/home/rthomp12/fewshot/mug_handle_20250103-220631_5'}


# #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~Teapot v Mug
parent_part_names = ['cup', 'handle']
child_part_names = ['body', 'tea_handle', 'spout', 'lid']

parent_part_model_files = {'cup': '/home/rthomp12/fewshot/mug_cup_20250103-220631_5',
                           'handle': '/home/rthomp12/fewshot/mug_handle_20250103-220631_5'}
child_part_model_files = {'body': '/home/rthomp12/fewshot/teapot_body_20250112-050420_5', 
                          'lid': '/home/rthomp12/fewshot/teapot_lid_20250112-050420_5',
                          'spout': '/home/rthomp12/fewshot/teapot_spout_20250112-050420_5',
                          'tea_handle': '/home/rthomp12/fewshot/teapot_handle_20250112-050420_5', }

parent_object = "mug"
# #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Mug v Watering Can
# parent_part_names = ['cup', 'handle']
# child_part_names = ['body', 'tea_handle', 'spout', ]

# parent_part_model_files = {'cup': '/home/rthomp12/fewshot/mug_cup_20250103-220631_5',
#                            'handle': '/home/rthomp12/fewshot/mug_handle_20250103-220631_5'}
# child_part_model_files = {'body': '/home/rthomp12/fewshot/part_based_warp_models/body_dict_20241031-012841_5', 
#                           'spout': '/home/rthomp12/fewshot/part_based_warp_models/spout_dict_20241031-012841_5',
#                           'tea_handle': '/home/rthomp12/fewshot/part_based_warp_models/handle_dict_20241031-012841_5', }

# parent_object = "mug"



# #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~Mug v Bowl

# child_part_names = ['whole_bowl']
# parent_part_names = ['cup', 'handle']


# child_part_model_files = {'whole_bowl': '/home/rthomp12/fewshot/part_based_warp_models/whole_bowl_20240426-000022_10'}
# parent_part_model_files = {'cup': '/home/rthomp12/fewshot/mug_cup_20250103-220631_5',
#                           'handle': '/home/rthomp12/fewshot/mug_handle_20250103-220631_5'}


parent_part_models = {part: CanonPart.from_pickle(parent_part_model_files[part]) for part in parent_part_names}
child_part_models = {part: CanonPart.from_pickle(child_part_model_files[part]) for part in child_part_names}



In [ ]:
# Verify the reconstruction and demo transformation
import copy as cp 

child_params = cp.deepcopy(warps['child_params'].item())
parent_params = cp.deepcopy(warps['parent_params'].item())
reconstructions = {}
transformed_scene_pcls = {}

child_part_names = ["spout"]
parent_part_names = ["cup"]


child_params['spout'].position, child_params['spout'].quat = utils.transform_to_pos_quat(np.matmul(
                                                            utils.pos_quat_to_transform(child_params['spout'].position, 
                                                                                   child_params['spout'].quat), 
                                                                            utils.pos_quat_to_transform(np.array([0.00,0,0]),                                                                                                                                         
                                                                            np.array([ -0.3360627, -0.0681233, -0.1866245, 0.9206478 ]))))
preplace_child_params = cp.deepcopy(warps['child_params'].item())
for part in child_part_names:
    preplace_child_params[part].position,  preplace_child_params[part].quat = utils.transform_to_pos_quat(np.matmul(ee_preplace, 
                                                                utils.pos_quat_to_transform(child_params[part].position, 
                                                                                   child_params[part].quat)))
for child_part in child_part_names:
    child_transform = utils.pos_quat_to_transform(child_params[child_part].position, child_params[child_part].quat)
    new_child_transform = np.matmul(ee_transform, child_transform)
    child_params[child_part].position, child_params[child_part].quat = \
        utils.transform_to_pos_quat(new_child_transform)

    transformed_scene_pcls[child_part] = utils.transform_pcd(scene_pcls[child_part],
                                                 ee_transform,
                                               )
    transformed_scene_pcls[f'preplace_{child_part}'] = utils.transform_pcd(scene_pcls[child_part],
                                                 ee_preplace,
                                               )
    reconstructions[f'reconstructed_{child_part}'] = \
        child_part_models[child_part].to_transformed_pcd(child_params[child_part])
    
    reconstructions[f'reconstructed_preplace_{child_part}'] = \
        child_part_models[child_part].to_transformed_pcd(preplace_child_params[child_part])
    
for parent_part in parent_part_names: 
    
    fix_transform = utils.pos_quat_to_transform(np.array([0.01,0.0,0.04]), np.array([0,0,0,1]))
    parent_params[parent_part].position, parent_params[parent_part].quat = \
        utils.transform_to_pos_quat(np.matmul(fix_transform, utils.pos_quat_to_transform(parent_params[parent_part].position, parent_params[parent_part].quat))) \

    transformed_scene_pcls[parent_part] = utils.transform_pcd(scene_pcls[parent_part],
                                                 fix_transform,
                                               )
    reconstructions[f'reconstructed_{parent_part}'] = \
        parent_part_models[parent_part].to_transformed_pcd(parent_params[parent_part])


grasp_point= np.atleast_2d(utils.transform_to_pos_quat(final_transform)[0])



camera = dict(
    eye=dict(x=1.2, y=-1.2, z=.2),
    center=dict(x=.8,y=.5,z=0)
)
viz_utils.show_pcds_plotly(transformed_scene_pcls|reconstructions|{'grasp': grasp_point}, camera=camera)

    

In [ ]:
# Find and save interaction points 

warps = np.load(f"{root_file}/initial_scene_warps.npz", allow_pickle=True)
warps = {k: warps[k] for k in warps.keys()}

nearby_points_delta = 0.03 # Empirically picked

(
    knns,
    deltas,
    target_indices,
) = demo.save_place_nearby_points_by_parts_v2(
    ["spout"],
    child_part_models,
    child_params,
    ["cup"],
    parent_part_models,
    parent_params,
    nearby_points_delta,
)
# print(deltas)
# for delta in deltas['spout']['cup']: 
#     delta[:,2] += -.03
knn_pickle_file = f'{root_file}/watering_can_interaction_points.pkl'
preplace_knn_pickle_file = f'{root_file}/watering_can_interaction_points.pkl'

# preplace_child_params = cp.deepcopy(warps['child_params'].item())
# for part in child_part_names:
#     preplace_child_params[part].position,  preplace_child_params[part].quat = utils.transform_to_pos_quat(np.matmul(ee_preplace, 
#                                                                 utils.pos_quat_to_transform(warps['child_params'].item()[part].position, 
#                                                                                    warps['child_params'].item()[part].quat)))

# nearby_points_delta = 0.15  
# (
#     preplace_knns,
#     preplace_deltas,
#     preplace_target_indices,
# ) = demo.save_place_nearby_points_by_parts_v2(
#     ["spout"],
#     child_part_models,
#     preplace_child_params,
#     ["cup"],
#     parent_part_models,
#     parent_params,
#     nearby_points_delta,
# )


interaction_points = {'knns': knns, 'deltas': deltas, "target_indices": target_indices}

# print(preplace_knns)
# preplace_interaction_points = {'knns': preplace_knns, 
#                                'deltas': preplace_deltas, 
                            #    "target_indices": preplace_target_indices}

pickle.dump(interaction_points, open(knn_pickle_file, 'wb'))
#pickle.dump(preplace_interaction_points, open(preplace_knn_pickle_file, 'wb'))


@@ 0.03723594235832555
spout
cup
# nearby points: 271499


In [6]:
# Visualize interaction points 

targets_child = {part: {} for part in child_part_names}
targets_parent = {part: {} for part in child_part_names}

for child_part in child_part_names:
    for parent_part in parent_part_names:
        if knns[child_part][parent_part] is None:
            continue
        anchors = child_part_models[child_part].to_pcd(child_params[child_part])[
            knns[child_part][parent_part]
        ]
        targets_child[child_part][parent_part] = np.mean(
            anchors + deltas[child_part][parent_part], axis=1
        )
        targets_parent[child_part][parent_part] = parent_part_models[
            parent_part
        ].to_pcd(parent_params[parent_part])[
            target_indices[child_part][parent_part]
        ] 

child_part_targets = {}   
child_targets_viz = {}
parent_targets_viz = {}

for child_part in child_part_names:
    child_part_transform  = utils.pos_quat_to_transform(child_params[child_part].position, 
                                                  child_params[child_part].quat)
    for parent_part in parent_part_names:
        if knns[child_part][parent_part] is None:
            continue
        parent_part_transform  = utils.pos_quat_to_transform(parent_params[parent_part].position, 
                                                  parent_params[parent_part].quat)
        child_targets_viz =  child_targets_viz | {
                             f'child_targets_{child_part}_{parent_part}': \
                             utils.transform_pcd(targets_child[child_part][parent_part],
                                                 child_part_transform),
                             }
        parent_targets_viz = parent_targets_viz | {
                             f'parent_targets_{child_part}_{parent_part}': \
                             utils.transform_pcd(targets_parent[child_part][parent_part],
                                                 parent_part_transform),
                             }

viz_pcls = scene_pcls | child_targets_viz | parent_targets_viz

viz_utils.show_pcds_plotly(viz_pcls)

In [ ]:
# create ndf_interface

from src.ndf_interface import NDFPartInterface
interface = NDFPartInterface(
        canon_source_parts_paths = child_part_model_files,
        canon_target_parts_paths = parent_part_model_files,
        source_part_names = child_part_names,
        target_part_names= parent_part_names,)


In [ ]:
# extract relevant part pair relationships

possible_relevant_part_pairs = []
for child_part in child_part_names:
    part_pairs = []
    for parent_part in parent_part_names:
        if knns[child_part][parent_part] is None:
            continue
        part_pairs.append((child_part, parent_part))
    possible_relevant_part_pairs.append(part_pairs)

possible_constraint_programs = list(itertools.product(*possible_relevant_part_pairs))
print(possible_constraint_programs)

[(('spout', 'cup'),)]


In [ ]:
# Make a prediction based on the training sample and calculate the distance between it and the ground-truth.
costs = []

child_parts = {name: scene_pcls[name] for name in child_part_names}
parent_parts = {name: scene_pcls[name] for name in parent_part_names}

for part_pair in possible_constraint_programs:
    print(part_pair)
    trans_predicted = interface.infer_relpose(
        child_parts, parent_parts, part_pair, se3=True, knn_pkl=knn_pickle_file
    )
    cost = utils.pose_distance(trans_predicted, ee_transform)
    costs.append(cost)
    print(cost)
    print()

min_cost = np.min(np.array(costs))
min_pair = possible_constraint_programs[np.argmin(np.array(costs))]

print(min_pair)
print(costs)
print(possible_constraint_programs)

(('spout', 'cup'),)
cup pose: [ 0.66894656 -0.08882692  0.04892848]


KeyboardInterrupt: 

In [ ]:
#save the interaction points